Giới thiệu và Khởi tạo Dữ liệuTrong phần này, chúng ta sẽ sử dụng một tập dữ liệu siêu nhỏ (nano-documents) được lấy trực tiếp từ tài liệu. Giả định rằng tất cả các từ đều được viết thường và loại bỏ dấu câu.

Truy vấn (Query): "sweet love"

Tài liệu (Documents):

- Doc 1: "Sweet sweet nurse! Love?"
- Doc 2: "Sweet sorrow"
- Doc 3: "How sweet is love?"
- Doc 4: "Nurse!"

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Khai báo tập tài liệu (Corpus) và làm sạch cơ bản
documents = [
    "sweet sweet nurse love", # Doc 1
    "sweet sorrow",           # Doc 2
    "how sweet is love",      # Doc 3
    "nurse"                   # Doc 4
]

# Truy vấn của người dùng
query = "sweet love"

Trong mô hình không gian vector (vector space model), tài liệu được biểu diễn dưới dạng các vector. Chúng ta không sử dụng số đếm từ vựng thô mà sử dụng trọng số tf-idf.Điểm liên quan (score) giữa truy vấn $q$ và tài liệu $d$ được tính bằng độ tương tự cosine (cosine similarity):

$$score(q,d) = cos(q,d) = \frac{q \cdot d}{|q| |d|}$$

In [2]:
# Khởi tạo mô hình TF-IDF
# (Lưu ý: scikit-learn sử dụng log tự nhiên thay vì log cơ số 10 như trong công thức 11.4 của sách,
# nhưng nguyên lý cơ bản là hoàn toàn tương đương).
vectorizer = TfidfVectorizer()

# Chuyển đổi tập tài liệu thành ma trận TF-IDF
tfidf_matrix = vectorizer.fit_transform(documents)

# Chuyển đổi câu truy vấn thành vector TF-IDF cùng không gian
query_vector = vectorizer.transform([query])

# Tính toán Cosine Similarity giữa truy vấn và tất cả các tài liệu
cosine_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

# Xếp hạng tài liệu dựa trên điểm Cosine
ranked_doc_indices = cosine_scores.argsort()[::-1]

print(f"Truy vấn: '{query}'\n")
print("Kết quả truy xuất (Xếp hạng từ cao xuống thấp):")
for rank, idx in enumerate(ranked_doc_indices):
    print(f"Top {rank+1} - Doc {idx+1} (Điểm Cosine: {cosine_scores[idx]:.4f}) | Nội dung: '{documents[idx]}'")

Truy vấn: 'sweet love'

Kết quả truy xuất (Xếp hạng từ cao xuống thấp):
Top 1 - Doc 1 (Điểm Cosine: 0.8354) | Nội dung: 'sweet sweet nurse love'
Top 2 - Doc 3 (Điểm Cosine: 0.5829) | Nội dung: 'how sweet is love'
Top 3 - Doc 2 (Điểm Cosine: 0.3385) | Nội dung: 'sweet sorrow'
Top 4 - Doc 4 (Điểm Cosine: 0.0000) | Nội dung: 'nurse'


Dựa trên kiến trúc tiêu chuẩn được mô tả trong hình 11.13, hệ thống RAG có hai giai đoạn: gọi bộ truy xuất (retriever) để lấy top-k tài liệu liên quan , sau đó tạo một prompt chứa câu truy vấn và các tài liệu này , rồi truyền nó cho một LLM (Generator).

RAG giúp giảm thiểu tình trạng "ảo giác" (hallucination) của LLM bằng cách cung cấp cho mô hình một tập hợp các tài liệu đáng tin cậy

In [3]:
# Lấy Top 2 tài liệu liên quan nhất từ bước Retrieval ở trên
TOP_K = 2
retrieved_passages = [documents[idx] for idx in ranked_doc_indices[:TOP_K]]

# Xây dựng Prompt theo đúng cấu trúc mô tả trong sách (Schematic of a RAG Prompt)
def build_rag_prompt(query, passages):
    prompt = ""
    for i, passage in enumerate(passages):
        prompt += f"retrieved passage {i+1}: {passage}\n"

    prompt += f"\nBased on these texts, answer this question: {query}?"
    return prompt

rag_prompt = build_rag_prompt(query, retrieved_passages)

print("=== MẪU PROMPT SẼ GỬI CHO LLM ===")
print(rag_prompt)
print("==================================")

# Mô phỏng hàm gọi LLM (Trong thực tế bạn sẽ dùng OpenAI API, HuggingFace, v.v.)
def mock_llm_generator(prompt):
    # LLM phân tích prompt và trả về câu trả lời dựa trên ngữ cảnh
    return "The texts suggest a strong thematic focus on 'sweet love', particularly associated with a nurse."

print("\nKết quả sinh văn bản từ LLM:")
print(mock_llm_generator(rag_prompt))

=== MẪU PROMPT SẼ GỬI CHO LLM ===
retrieved passage 1: sweet sweet nurse love
retrieved passage 2: how sweet is love

Based on these texts, answer this question: sweet love?

Kết quả sinh văn bản từ LLM:
The texts suggest a strong thematic focus on 'sweet love', particularly associated with a nurse.
